# **Data Collection and Transformation**


In [16]:
import pandas as pd
import numpy as np
import wikipedia as wiki
import requests as rq
import bs4 as soup
import io
import re

## **Collecting the Data**
We start by accessing the Paleobiology Database through their [data service API](https://paleobiodb.org/data1.2/). The criteria I am using is based off the Taxonomy of Fossil Occurrences Dataset. Cleaning up the data leaves us with the Classifications, Diet, and First/Last Appearances in the fossil record.

In [17]:
taxa_url = rq.get('https://paleobiodb.org/data1.2/occs/taxa.csv?base_name=Dinosauria&idreso=species&idqual=certain&pres=regular&max_ma=252&min_ma=65&show=class,size,app,ecospace,img').content
taxa = pd.read_csv(io.StringIO(taxa_url.decode('utf-8')))[['taxon_rank', 'taxon_name', 'genus', 'family', 'taxon_size', 'diet', 'firstapp_max_ma', 'lastapp_min_ma']]

taxa = taxa.dropna(subset=['taxon_name']).query('(taxon_rank == \'genus\') or (taxon_rank == \'species\')')
taxa.columns = ['Rank', 'Name', 'Genus', 'Family', 'Taxon Size', 'Diet', 'Max MYA', 'Min MYA']

taxa = taxa.replace(regex=['NO_FAMILY_SPECIFIED'], value='')

taxa['Diet'] = taxa['Diet'].str.capitalize()

taxa.head()

,Rank,Name,Genus,Family,Taxon Size,Diet,Max MYA,Min MYA
10,genus,Ajkaceratops,Ajkaceratops,,2.0,Herbivore,86.3,83.6
11,species,Ajkaceratops kozmai,Ajkaceratops,,1.0,Herbivore,86.3,83.6
12,genus,Turanoceratops,Turanoceratops,,2.0,Herbivore,93.9,89.8
13,species,Turanoceratops tardabilis,Turanoceratops,,1.0,Herbivore,93.9,89.8
14,genus,Zuniceratops,Zuniceratops,,2.0,Herbivore,93.9,89.8


While the dataset above contains a surplus of information, it does not include locations of where the species have been found. To account for this, I am grabbing another dataset that contains all the known fossil occurrences and their respective origin location. We need this additional dataset to analyze the geographical distribution of fossil occurrences in the Mesozoic Era

In [18]:
# Scraping data from the Paleobiology Database and cleaning up the Null values
occ_url = rq.get('https://paleobiodb.org/data1.2/occs/list.csv?base_name=Dinosauria&taxon_reso=species&idqual=certain&pres=regular&max_ma=252&min_ma=65&show=class,coords,loc,strat,acconly').content
occ = pd.read_csv(io.StringIO(occ_url.decode('utf-8')))[['accepted_name', 'lng', 'lat', 'formation', 'cc', 'state', 'county', 'collection_no']]

occ.columns = ['Species', 'Longitude', 'Latitude', 'Formation', 'Country', 'State', 'County', 'Collection']

occ.head()

,Species,Longitude,Latitude,Formation,Country,State,County,Collection
0,Chaoyangsaurus youngi,123.966698,42.933300,Tuchengzi,CN,Liaoning,Chaoyang,10755
1,Protarchaeopteryx robusta,120.733330,41.799999,Yixian,CN,Liaoning,NaN,10764
2,Caudipteryx zoui,120.733330,41.799999,Yixian,CN,Liaoning,NaN,10764
3,Gorgosaurus libratus,-111.528732,50.740726,Dinosaur Park,CA,Alberta,NaN,11890
4,Gorgosaurus libratus,-111.549347,50.737015,Dinosaur Park,CA,Alberta,NaN,11893


## **Cleaning the Data (Taxa)**
With the both of these dataframes at our disposal, the next step is to clean it up in a presentable format. Firstly, we want to fix the null values in the Age Columns and add in the corresponding Period and Epochs. To do this, we can sort by year and then choose the Period and Epoch based on what Age the species or genus lived in.

In [29]:
# Looking at the dataframe, we need to clean the columns relating to species lifetime --> Some dinosaurs have NaN as their entries for Early and Late Ages
periods = ['Triassic', 'Jurassic', 'Cretaceous']
epochs = ['Lower', 'Middle', 'Upper']


tri_ages = ['Induan', 'Olenekian', 'Anisian', 'Ladinian', 'Carnian', 'Norian', 'Rhaetian']
jur_ages = ['Hettangian', 'Sinemurian', 'Pliensbachian', 'Toarcian', 'Aalenian', 'Bajocian', 'Bathonian', 'Callovian', 'Oxfordian', 'Kimmeridgian', 'Tithonian']
cre_ages = ['Berriasian', 'Valanginian', 'Hauterivian', 'Barremian', 'Aptian', 'Albian', 'Cenomanian', 'Turonian', 'Coniacian', 'Santonian', 'Campanian', 'Maastrichtian']

low_ep = ['Induan', 'Olenekian', 'Hettangian', 'Sinemurian', 'Pliensbachian', 'Toarcian', 'Berriasian', 'Valanginian', 'Hauterivian', 'Barremian', 'Aptian', 'Albian']
mid_ep = ['Anisian', 'Ladinian', 'Aalenian', 'Bajocian', 'Bathonian', 'Callovian']
upp_ep = ['Carnian', 'Norian', 'Rhaetian', 'Oxfordian', 'Kimmeridgian', 'Tithonian', 'Cenomanian', 'Turonian', 'Coniacian', 'Santonian', 'Campanian', 'Maastrichtian']


ages = [*tri_ages, *jur_ages, *cre_ages]

# Arguements for Age
mya_args = lambda x : [(taxa[x] <= 251.9) & (taxa[x] > 251.2), 
                (taxa[x] <= 251.2) & (taxa[x] > 247.2),
                (taxa[x] <= 247.2) & (taxa[x] > 242),
                (taxa[x] <= 242) & (taxa[x] > 237),
                (taxa[x] <= 237) & (taxa[x] > 227),
                (taxa[x] <= 227) & (taxa[x] > 208.5),
                (taxa[x] <= 208.5) & (taxa[x] > 201.4),
                (taxa[x] <= 201.4) & (taxa[x] > 199.5),
                (taxa[x] <= 199.5) & (taxa[x] > 192.9),
                (taxa[x] <= 192.9) & (taxa[x] > 184.2),
                (taxa[x] <= 184.2) & (taxa[x] > 174.7),
                (taxa[x] <= 174.7) & (taxa[x] > 170.9),
                (taxa[x] <= 170.9) & (taxa[x] > 168.2),
                (taxa[x] <= 168.2) & (taxa[x] > 165.3),
                (taxa[x] <= 165.3) & (taxa[x] > 161.5),
                (taxa[x] <= 161.5) & (taxa[x] > 154.8),
                (taxa[x] <= 154.8) & (taxa[x] > 149.2),
                (taxa[x] <= 149.2) & (taxa[x] > 145),
                (taxa[x] <= 145) & (taxa[x] > 139.8),
                (taxa[x] <= 139.8) & (taxa[x] > 132.6),
                (taxa[x] <= 132.6) & (taxa[x] > 125.77),
                (taxa[x] <= 125.77) & (taxa[x] > 121.4),
                (taxa[x] <= 121.4) & (taxa[x] > 113),
                (taxa[x] <= 113) & (taxa[x] > 100.5),
                (taxa[x] <= 100.5) & (taxa[x] > 93.9),
                (taxa[x] <= 93.9) & (taxa[x] > 89.8),
                (taxa[x] <= 89.8) & (taxa[x] > 86.3),
                (taxa[x] <= 86.3) & (taxa[x] > 83.6),
                (taxa[x] <= 83.6) & (taxa[x] > 72.1),
                (taxa[x] <= 72.1) & (taxa[x] > 66)] 
                
# Arguments for Period and Epoch (will combine into one column later)
pers = lambda x : [(taxa[x].isin(tri_ages)),
                   (taxa[x].isin(jur_ages)),
                   (taxa[x].isin(cre_ages))]

eps = lambda x: [(taxa[x].isin(low_ep)),
                 (taxa[x].isin(mid_ep)),
                 (taxa[x].isin(upp_ep))]


# Adding the Period and Age columns
taxa['Early Age'] = np.select(mya_args('Max MYA'), ages, default=pd.NaT)

# We add 0.01 to accomodate for edge cases where a dinosaur is estimated to have lived at the cusp of two mesozoic ages
taxa['Min MYA'] += 0.01
taxa['Late Age'] = np.select(mya_args('Min MYA'), ages, default=pd.NaT)
taxa['Min MYA'] -= 0.01
taxa['Late Age'] = taxa['Late Age'].fillna(taxa['Early Age'])

taxa['Early Period'] = np.select(eps('Early Age'), epochs, default=pd.NaT) + ' ' + np.select(pers('Early Age'), periods, default=pd.NaT)
taxa['Late Period'] = np.select(eps('Late Age'), epochs, default=pd.NaT) + ' ' + np.select(pers('Late Age'), periods, default=pd.NaT)

# Adding a lifespan column to show how long each species/genus lived
taxa['Lifespan (MYA)'] = taxa['Max MYA'] - taxa['Min MYA']
taxa.head()

,Rank,Name,Genus,Family,Taxon Size,Diet,Max MYA,Min MYA,Early Age,Late Age,Early Period,Late Period,Lifespan (MYA)
10,genus,Ajkaceratops,Ajkaceratops,,2.0,Herbivore,86.3,83.6,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,2.7
11,species,Ajkaceratops kozmai,Ajkaceratops,,1.0,Herbivore,86.3,83.6,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,2.7
12,genus,Turanoceratops,Turanoceratops,,2.0,Herbivore,93.9,89.8,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,4.1
13,species,Turanoceratops tardabilis,Turanoceratops,,1.0,Herbivore,93.9,89.8,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,4.1
14,genus,Zuniceratops,Zuniceratops,,2.0,Herbivore,93.9,89.8,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,4.1


Next, we should separate the Genera and Species from each other for more efficient tracking. The dataset was designed to include the Genus in the taxon size count, so we will have to account for that when we separate the Genus.

In [20]:
species = taxa.loc[taxa['Rank'] == 'species'].reset_index().drop(columns=['Rank', 'Taxon Size', 'Family', 'index'])
genus = taxa.loc[taxa['Rank'] == 'genus'].reset_index().drop(columns=['Rank', 'Genus', 'index'])

# Dropping this count by 1 because because the genus in the original dataframe was counted towards the taxon size
genus['Taxon Size'] = genus['Taxon Size'].astype(int) - 1

genus.head()

,Name,Family,Taxon Size,Diet,Max MYA,Min MYA,Early Age,Late Age,Early Period,Late Period,Lifespan (MYA)
0,Ajkaceratops,,1,Herbivore,86.3,83.6,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,2.7
1,Turanoceratops,,1,Herbivore,93.9,89.8,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,4.1
2,Zuniceratops,,1,Herbivore,93.9,89.8,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,4.1
3,Bagaceratops,Protoceratopsidae,1,Herbivore,83.6,66.0,Campanian,Maastrichtian,Upper Cretaceous,Upper Cretaceous,17.6
4,Breviceratops,Protoceratopsidae,1,Herbivore,83.6,72.1,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,11.5


***Some things to note are that several of the generas in the data set are either informally named, described in a way that would not be recognized in a web scrape, do not have enough research to assign them to a specific clade, or are trace/egg fossils (which get their own separate genera). Thus, we need to clean these cells up and add some more indentification columns. The following lines acknowledge these discrepancies:***

In [21]:
nondinos = ['Krzyzanowskisaurus', 'Actiosaurus', 'Volgavis', 'Yacoraitichnus', 'Yezosaurus', 'Brasileosaurus', 'Subtiliolithus', 'Fuscinapedis', 'Himeoolithus', 'Protoavis', 'Salfitichnus', 'Nyasasaurus']
no_webpage = ['Crateropteryx', 'Bellulornis', 'Alethoalaornis', 'Eopengornis', 'Fortunguavis', 'Gretcheniao', 'Holbotia', 'Linyiornis', 'Microenantiornis', 'Monoenantiornis', 'Otogornis', 'Pterygornis', 'Shangyang', 'Horezmavis', 'Platanavis']

# Adding columns for informal dinosaurs
genus['Informal'] = False

# Renaming mislabelled dinosaurs
genus.at[genus.index[genus['Name'] == 'Megalosaurus (Poekilopleuron)'].values[0], 'Name'] = 'Poekilopleuron'
genus.at[genus.index[genus['Name'] == 'Jingia'].values[0], 'Name'] = 'Jingiella'
genus.at[genus.index[genus['Name'] == 'Bellulia'].values[0], 'Name'] = 'Bellulornis'

# Removing genera that are informally named, are trace/egg fossils, or don't have web pages to get data from
for i in nondinos + no_webpage:
    try:
        x = genus.index[genus['Name'] == i].values[0]
        genus = genus.drop(axis=0, index=x)
    except:
        pass
        

genus = genus.reset_index()


Since we’ve cleaned up the genera dataset, we need to create a helper function to fetch the raw webpage data for each dinosaur genus. Some taxa might redirect to entirely different pages during the search or are informally named (as discussed before), so we’ll need to handle those edge cases manually. Once the webpage returns, we will return the bio-classification table data in array form. Once the function is ready, we’ll apply need to create another function to parse this data to be readable.

In [22]:
# Helper function that finds the corresponding wikipedia page for each dinosaur
def get_webpage(genus):
    
    # Edge case when looking up Qianlong and Wulong since they redirect to different pages
    if genus == 'Qianlong':
        return get_webpage('Qianlong shouhu')
    elif genus == 'Wulong':
        return get_webpage('Wulong bohaiensis')
   
   # Web scraping each genus name to figure out classification
    try:
        page = f'https://en.wikipedia.org/wiki/{genus}'
        response = rq.get(page)
      
        biota = soup.BeautifulSoup(response.text, 'html.parser')
        
        # Checking for informal dinosaurs
        if ('informally named dinosaurs' in biota.find('title').text) or (genus in ['Tiantaisaurus', 'Khanazeem']):
            return 'Informally Named Dinosaur'
        else:
            biota = biota.find('table', {'class': 'infobox biota'}).find_all('tr')
      
    except:
      
      # Covering case where the genus is named after an existing topic
        try:
            page = f'https://en.wikipedia.org/wiki/{genus}_(dinosaur)'
            response = rq.get(page)
      
            biota = soup.BeautifulSoup(response.text, 'html.parser').find('table', {'class': 'infobox biota'}).find_all('tr')
      
        except:
            pass
   
    return biota

# get_webpage('Wulong')


Now that we have the raw data, we need to format it into a readable data frame. Using the results from the helper function, we’ll process each line to decide what to include, focusing on a modified version of Michael Benton’s classification. This approach uses Linnaean principles to refine dinosaur taxonomy.

However, for Saurischian infraorders, we’ll need to simplify things. The Linnaean structure doesn’t cleanly apply to groups like Theropoda and Sauropodamorpha due to the complexity of clades within Theropoda and the relative lack of diversity within Sauropodamorpha. Instead, we’ll categorize these groups as Titanosauria and Sauropoda for Sauropodomorpha, and Avialae and Non-Avian for Theropoda, focusing on comparisons that fit this dataset’s scope.

In [23]:
orders = ['Ornithischia', 'Saurischia']
suborders = ['Neornithischia', 'Thyreophora', 'Sauropodomorpha', 'Theropoda']
infraorders = ['Ornithopoda', 'Ceratopsia', 'Pachycephalosauria', 'Stegosauria', 'Ankylosauria', 'Sauropoda', 'Titanosauria', 'Avialae', 'Non-Avian']

# Creating a dictionary to use when the dinosaur is noted to be avian
avian = {'Order': orders[1], 'Suborder': suborders[3], 'Infraorder': infraorders[7], 'Family': ''}

def wiki_scrape_genus(biota):
   # Defining the dictionary that will be put into the dataframe
   data = {'Order':'', 'Suborder':'', 'Infraorder':'', 'Family':''}
   
   # iterating through rows that feature classification terms
   for row in biota:
      biota_data = row.find_all('td')
      
      
      try:
         bio_class = re.match(r'([A-Z][a-z]*)', biota_data.pop(0).text.strip())[0]
         
         # Only iterating through Clades, Families, and Genera
         if bio_class in ['Clade', 'Family', 'Genus']:
            bio_name = re.match(r'([^A-Za-z][A-Z][a-z]*)|([A-Z][a-z]*)', biota_data.pop(0).text.strip())[0]
            bio_name = re.sub('†', '', bio_name)
            
            # Setting values for Order, Suborder, and Infraorder
            if bio_class in ['Clade', 'Superfamily']:
               if bio_name in orders:
                  data['Order'] = bio_name
                  
               elif bio_name in suborders:
                  data['Suborder'] = bio_name
                  
                  # Setting Default Value for Theropods to be non-avian
                  if bio_name == 'Theropoda':
                     data['Infraorder'] = 'Non-Avian'

                  # Setting Default Value for Sauropods to be Sauropoda 
                  elif bio_name == 'Sauropodamorpha':
                     data['Infraorder'] = 'Sauropoda'      
                     
                                 
               elif bio_name in infraorders:
                  data['Infraorder'] = bio_name
                     
               # Including edge case of Euornithes since they are also Late Mesozoic Avians
               elif bio_name == 'Euornithes':
                  data = avian
                  
         
            elif bio_class == 'Family':
               data['Family'] = bio_name                         
               
         # In the event that the wikipedia page is not labelled with Dinosauria
         if bio_class == 'Class':
            bio_name = re.match(r'([^A-Za-z][A-Z][a-z]*)|([A-Z][a-z]*)', biota_data.pop(0).text.strip())[0]
            
            if bio_name == 'Aves':
               data = avian       
      
      except:
         pass
      
   return data

We’ve created the helper functions to gather and parse the data correctly, so the next step is to apply them to each row. However, some genera may return errors during this process. If that happens, we’ll need to fall back on running a web search for the Family listed in the original PaleoDB dataset, provided that information is available.

In [24]:
genus['Order'] = ''
genus['Suborder'] = ''
genus['Infraorder'] = ''

genus = genus[['Name', 'Family', 'Infraorder', 'Suborder', 'Order', 'Informal', 'Taxon Size', 'Diet', 'Max MYA', 'Min MYA', 'Lifespan (MYA)', 'Early Age', 'Late Age', 'Early Period', 'Late Period']]

for i, row in genus.iterrows():
    
    try:
        dino = genus.iloc[i]['Name']
        
        biota = get_webpage(dino)
        if (biota == 'Informally Named Dinosaur'):
            genus.at[i, 'Informal'] = True
        else:
            wiki_data = wiki_scrape_genus(biota)
            for col in wiki_data:
                genus.loc[i, col] = wiki_data[col]
                
    except:
        if genus.iloc[i]['Family'] != '':
            try:
                fam = genus.iloc[i]['Family']
                biota = get_webpage(dino)
                wiki_data = wiki_scrape_genus(biota)
                for col in wiki_data:
                    genus.loc[i, col] = wiki_data[col]
            except:
                pass


With the list of genera complete, we can move on to merging the dataframes. The species dataframe will be combined with both the genera and fossil occurrences dataframes, creating a more comprehensive dataset, then the occurrences dataframe will join the species. These merged datasets will allow us to analyze and compare species, genera, and their respective fossils more effectively.

In [25]:
species = species.merge(genus[['Name', 'Family', 'Infraorder', 'Suborder', 'Order', 'Informal']], left_on='Genus', right_on='Name', how='left').drop('Name_y', axis=1).rename(columns={'Name_x':'Species'})
species = species[['Species', 'Genus', 'Family', 'Infraorder', 'Suborder', 'Order', 'Informal', 'Diet', 'Early Age', 'Late Age', 'Early Period', 'Late Period', 'Max MYA', 'Min MYA', 'Lifespan (MYA)',]]

occ = occ.merge(species, on='Species', how='left').drop('Informal', axis=1)

#species = species[species['Informal' != True]].drop(columns='Informal')

species.head()

,Species,Genus,Family,Infraorder,Suborder,Order,Informal,Diet,Early Age,Late Age,Early Period,Late Period,Max MYA,Min MYA,Lifespan (MYA)
0,Ajkaceratops kozmai,Ajkaceratops,,,,Ornithischia,False,Herbivore,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,86.3,83.6,2.7
1,Turanoceratops tardabilis,Turanoceratops,,,,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
2,Zuniceratops christopheri,Zuniceratops,,,,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
3,Bagaceratops rozhdestvenskyi,Bagaceratops,Protoceratopsidae,,,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5
4,Breviceratops kozlowskii,Breviceratops,Protoceratopsidae,,,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5


In [26]:
species = species.loc[species['Informal'] != True].drop(columns=['Informal'])
species.head()


,Species,Genus,Family,Infraorder,Suborder,Order,Diet,Early Age,Late Age,Early Period,Late Period,Max MYA,Min MYA,Lifespan (MYA)
0,Ajkaceratops kozmai,Ajkaceratops,,,,Ornithischia,Herbivore,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,86.3,83.6,2.7
1,Turanoceratops tardabilis,Turanoceratops,,,,Ornithischia,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
2,Zuniceratops christopheri,Zuniceratops,,,,Ornithischia,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
3,Bagaceratops rozhdestvenskyi,Bagaceratops,Protoceratopsidae,,,Ornithischia,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5
4,Breviceratops kozlowskii,Breviceratops,Protoceratopsidae,,,Ornithischia,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5


This merge is an essential step in refining our dataset, as it combines taxonomic details with the broader context of fossil occurrences. With the finalized dataframe, we’ll be ready to analyze patterns, compare species distributions, and uncover deeper insights into paleobiological history. From here, we can proceed to the visualization phase of this Mesozoic analysis.

In [27]:
genus.to_csv('../data-reserve/genera-list.csv', index=False)
species.to_csv('../data-reserve/species-list.csv', index=False)